# 6 WorkFlow Gerencial, futuro=SEP

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente (Google Colab o Jupyter remoto)

**Google Colab:** correr con runtime **Python 3**, luego cambiar a **R**.

**Jupyter remoto:** correr las celdas de setup con kernel **Python** (o saltear la celda de Drive si no está en Colab). El resto del workflow corre en **R**.

Las rutas se adaptan solas: Colab usa `/content/...`, Jupyter remoto usa `~/labo1` (o la variable de entorno `LABO_BASE`).

**Solo Colab:** conectar Google Drive para persistencia de archivos.

**Jupyter remoto:** esta celda detecta el entorno y saltea el mount si no hay `google.colab`.

In [ ]:
# Setup de entorno: Colab monta Drive, Jupyter remoto usa ~/labo1
import os

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/.drive")
    print("Entorno: Google Colab")
except ImportError:
    print("Entorno: Jupyter remoto (sin google.colab)")

if IN_COLAB:
    os.environ["LABO_BASE"] = "/content/buckets/b1"
    os.environ["LABO_DATASETS"] = "/content/datasets"
else:
    labo_base = os.environ.get("LABO_BASE", os.path.expanduser("~/labo1"))
    os.environ["LABO_BASE"] = labo_base
    os.environ["LABO_DATASETS"] = os.path.join(labo_base, "datasets")
    print("LABO_BASE =", labo_base)
    print("LABO_DATASETS =", os.environ["LABO_DATASETS"])

Crea carpetas, configura `kaggle.json` y descarga datasets.

**Colab:** copiar `kaggle.json` a `My Drive/labo1/kaggle/` antes del primer arranque.

**Jupyter remoto:** copiar `kaggle.json` a `~/labo1/kaggle/kaggle.json` (o dejarlo en la raíz del repo; el script lo busca). Opcional: exportar `LABO_BASE=/ruta/a/tu/carpeta`.



In [ ]:
%%bash

set -euo pipefail

if [ -d "/content/.drive" ]; then
  echo "Setup Colab"
  mkdir -p "/content/.drive/My Drive/labo1"
  mkdir -p "/content/buckets"
  ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1
  LABO_BASE="/content/buckets/b1"
  LABO_DATASETS="/content/datasets"
else
  echo "Setup Jupyter remoto"
  LABO_BASE="${LABO_BASE:-$HOME/labo1}"
  LABO_DATASETS="${LABO_DATASETS:-$LABO_BASE/datasets}"
fi

mkdir -p "$LABO_BASE/exp"
mkdir -p "$LABO_BASE/datasets"
mkdir -p "$LABO_DATASETS"
mkdir -p ~/.kaggle

KAGGLE_JSON=""
for candidate in \
  "$LABO_BASE/kaggle/kaggle.json" \
  "$HOME/kaggle.json" \
  "$HOME/Master/LABO1/kaggle.json" \
  "$(pwd)/kaggle.json" \
  "$(pwd)/../../kaggle.json" \
  "$(pwd)/../../../kaggle.json"
do
  if [ -f "$candidate" ]; then
    KAGGLE_JSON="$candidate"
    break
  fi
done

if [ -n "$KAGGLE_JSON" ]; then
  cp "$KAGGLE_JSON" ~/.kaggle/kaggle.json
  chmod 600 ~/.kaggle/kaggle.json
  echo "kaggle.json instalado desde $KAGGLE_JSON"
else
  echo "AVISO: no se encontro kaggle.json"
fi

descargar() {
  carpeta_destino="$LABO_BASE/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if [ ! -f "$carpeta_destino$archivo" ]; then
    wget "$url_origen$archivo" -O "$carpeta_destino$archivo"
  fi

  if [ "$LABO_DATASETS" != "$carpeta_destino" ] && [ ! -f "$LABO_DATASETS/$archivo" ]; then
    cp "$carpeta_destino$archivo" "$LABO_DATASETS/$archivo"
  fi
}

descargar "dataset_pequeno.csv"
descargar "gerencial_competencia_2026.csv.gz"

echo "LABO_BASE=$LABO_BASE"
echo "LABO_DATASETS=$LABO_DATASETS"


## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 102191

PARAM$experimento <- 6300
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# rutas: Colab /content, Jupyter remoto ~/labo1
if (dir.exists("/content/buckets/b1")) {
  PARAM$paths$base <- "/content/buckets/b1"
  PARAM$paths$datasets <- "/content/datasets"
} else {
  PARAM$paths$base <- path.expand(Sys.getenv("LABO_BASE", "~/labo1"))
  PARAM$paths$datasets <- file.path(PARAM$paths$base, "datasets")
}

PARAM$kaggle$competencia <- "labo-1-ros-2026-manager"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

PARAM$montecarlo <- list(
  n_semillas = 7,
  semillas = PARAM$semilla_primigenia + 1:7,
  top_configs = length(PARAM$kaggle$cortes),
  min_cor_predicciones = 0.80,
  min_overlap_topk = 0.80
)

PARAM$paths

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd(file.path(PARAM$paths$base, "exp"))
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd(file.path(PARAM$paths$base, "exp", experimento_folder))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(file.path(PARAM$paths$datasets, PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}

# ============================================================
# EXPERIMENTO: Trend3 + Trend6 relativo
# Se crean tendencias temporales normalizadas:
# trend3 = (actual - lag3) / (abs(lag3) + 1)
# trend6 = (actual - lag6) / (abs(lag6) + 1)
# ============================================================

dataset[, paste0(cols_lagueables, "_lag3") := shift(.SD, 3, NA, "lag"),
        by = numero_de_cliente,
        .SDcols = cols_lagueables]

dataset[, paste0(cols_lagueables, "_lag6") := shift(.SD, 6, NA, "lag"),
        by = numero_de_cliente,
        .SDcols = cols_lagueables]

for (vcol in cols_lagueables)
{
  dataset[, paste0(vcol, "_trend3") :=
            (get(vcol) - get(paste0(vcol, "_lag3"))) /
            (abs(get(paste0(vcol, "_lag3"))) + 1)]

  dataset[, paste0(vcol, "_trend6") :=
            (get(vcol) - get(paste0(vcol, "_lag6"))) /
            (abs(get(paste0(vcol, "_lag6"))) + 1)]
}

# Se eliminan lags auxiliares para aislar el efecto de trend3/trend6
dataset[, paste0(cols_lagueables, "_lag3") := NULL]
dataset[, paste0(cols_lagueables, "_lag6") := NULL]

gc()


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

# se sacan los meses de pandemia (marzo a junio 2020); en gerencial solo existen 202005 y 202006
PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007
)

PARAM$trainingstrategy$training_pct <- 0.3


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

Esta celda tarda en correr interminables 7 minutos en Colab
<br> ya que debe instalar la librería de LightGBM

In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE,
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  num_iterations= 2048,
  early_stopping_rounds= 200,
  # --- valores default, pisados por el grid ---
  learning_rate    = 0.05,
  feature_fraction = 0.5,
  bagging_fraction = 0.8,
  bagging_freq     = 1,
  lambda_l1        = 0,
  lambda_l2        = 0,
  num_leaves       = 64,
  min_data_in_leaf = 128
)


In [ ]:
# Estima AUC en validation para una combinacion de hiperparametros
# Acepta: num_leaves, min_data_in_leaf, learning_rate,
#         feature_fraction, lambda_l2  (ademas de los fijos)

Estimar_AUC_lightgbm <- function(x) {

  param_completo <- modifyList(PARAM$lgbm$param_fijos, as.list(x))

  modelo_train <- lgb.train(
    data   = dtrain,
    valids = list(valid = dvalidate),
    eval   = "auc",
    param  = param_completo,
    verbose= -100
  )

  AUC   <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]
  niter <- modelo_train$best_iter

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", niter,
    " AUC ",   AUC
  )

  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}


seteo del Grid Search

In [ ]:
# grid corto (~30 combos) para una corrida rapida; mantiene las 6 columnas que usa el resto del pipeline
tb_nueva <- CJ(
  num_leaves        = c(255, 511, 1023),
  min_data_in_leaf  = c(20, 50, 100),
  learning_rate     = c(0.03),
  feature_fraction  = c(0.5, 0.8),
  lambda_l2         = c(0, 10),
  min_gain_to_split = c(0)
)
tb_nueva <- tb_nueva[min_data_in_leaf <= num_leaves * 4]

cat("Combinaciones:", nrow(tb_nueva), "\n")


Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 50 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [ ]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

#### 6.3.2.3  Validación Monte Carlo

Las 11 mejores configs se validan con múltiples semillas. `tb_estabilidad` tiene **11 filas** con métricas y gaps vs umbrales. Las predicciones promedio van en `lst_pred_promedio` (aparte).

Si ninguna pasa todos los umbrales, el reporte muestra cuán cerca quedó cada una. Producción usa fallback: config con mayor `cor_min`.


In [ ]:
overlap_topk <- function(pred_a, pred_b, clientes, k) {
  top_a <- clientes[order(-pred_a)][1:k]
  top_b <- clientes[order(-pred_b)][1:k]
  length(intersect(top_a, top_b)) / k
}

estabilidad_preds <- function(preds, clientes, cortes) {
  n <- length(preds)
  cors <- c()
  overlaps <- c()

  for (i in 1:(n - 1)) {
    for (j in (i + 1):n) {
      cors <- c(cors, cor(preds[[i]], preds[[j]]))
      for (k in cortes) {
        overlaps <- c(overlaps, overlap_topk(preds[[i]], preds[[j]], clientes, k))
      }
    }
  }

  list(
    cor_min = min(cors),
    cor_mean = mean(cors),
    overlap_min = min(overlaps),
    overlap_mean = mean(overlaps)
  )
}

entrenar_y_predecir <- function(hiper, semilla, dfinal_train, dfuture, campos_buenos) {
  fijos <- copy(PARAM$lgbm$param_fijos)
  fijos$num_iterations <- NULL
  fijos$early_stopping_rounds <- NULL
  param_run <- modifyList(fijos, hiper)
  param_run$seed <- semilla

  modelo <- lgb.train(
    data = dfinal_train,
    param = param_run,
    verbose = -100
  )

  pred <- predict(
    modelo,
    data.matrix(dfuture[, campos_buenos, with = FALSE])
  )

  list(modelo = modelo, pred = pred)
}

##### Datasets para Monte Carlo


In [ ]:
# se sacan los meses de pandemia (marzo a junio 2020); en gerencial solo existen 202005 y 202006
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

dfinal_train <- lgb.Dataset(
  data = data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with = FALSE]),
  label = dataset[fold_final_train == TRUE, clase01],
  free_raw_data = TRUE
)

nrow(dfinal_train)

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[foto_mes %in% PARAM$trainingstrategy$future]
nrow(dfuture)

##### Corrida Monte Carlo

por favor no se asuste con los warnings que pudieran aparecer


In [ ]:
setorder(tb_nueva, -AUC)

tb_candidatas <- tb_nueva[1:min(PARAM$montecarlo$top_configs, .N)]
tb_candidatas[, config_rank := .I]
tb_candidatas[, corte_asignado := PARAM$kaggle$cortes[config_rank]]

resultados_mc <- lapply(1:nrow(tb_candidatas), function(i) {
  hiper <- as.list(tb_candidatas[i, .(num_leaves, min_data_in_leaf, learning_rate, feature_fraction, lambda_l2, num_iterations)])

  cat(format(Sys.time(), "%X"), " MC config", i, "/", nrow(tb_candidatas),
      " rank=", tb_candidatas[i, config_rank],
      " leaves=", hiper$num_leaves,
      " min_data=", hiper$min_data_in_leaf, "\n")

  preds <- lapply(PARAM$montecarlo$semillas, function(s) {
    entrenar_y_predecir(hiper, s, dfinal_train, dfuture, campos_buenos)$pred
  })

  pred_promedio <- Reduce(`+`, preds) / length(preds)
  est <- estabilidad_preds(preds, dfuture$numero_de_cliente, PARAM$kaggle$cortes)

  corte_k <- min(tb_candidatas[i, corte_asignado], length(pred_promedio))
  n_sem <- length(preds)
  overlaps_corte <- c()
  for (si in 1:(n_sem - 1)) {
    for (sj in (si + 1):n_sem) {
      overlaps_corte <- c(overlaps_corte,
        overlap_topk(preds[[si]], preds[[sj]], dfuture$numero_de_cliente, corte_k)
      )
    }
  }
  overlap_corte_min <- min(overlaps_corte)

  um_cor <- PARAM$montecarlo$min_cor_predicciones
  um_ovl <- PARAM$montecarlo$min_overlap_topk

  pasa_cor <- est$cor_min >= um_cor
  pasa_overlap <- est$overlap_min >= um_ovl
  pasa_corte <- overlap_corte_min >= um_ovl

  list(
    fila = data.table(
      config_rank = tb_candidatas[i, config_rank],
      corte_asignado = tb_candidatas[i, corte_asignado],
      num_leaves = tb_candidatas[i, num_leaves],
      min_data_in_leaf = tb_candidatas[i, min_data_in_leaf],
      learning_rate = tb_candidatas[i, learning_rate],
      feature_fraction = tb_candidatas[i, feature_fraction],
      lambda_l2 = tb_candidatas[i, lambda_l2],
      num_iterations = tb_candidatas[i, num_iterations],
      AUC = tb_candidatas[i, AUC],
      cor_min = est$cor_min,
      cor_mean = est$cor_mean,
      overlap_min = est$overlap_min,
      overlap_mean = est$overlap_mean,
      overlap_corte_min = overlap_corte_min,
      umbral_cor = um_cor,
      umbral_overlap = um_ovl,
      gap_cor = est$cor_min - um_cor,
      gap_overlap = est$overlap_min - um_ovl,
      gap_overlap_corte = overlap_corte_min - um_ovl,
      pasa_cor = pasa_cor,
      pasa_overlap = pasa_overlap,
      pasa_corte = pasa_corte,
      estable = pasa_cor && pasa_overlap && pasa_corte
    ),
    pred_promedio = pred_promedio
  )
})

tb_estabilidad <- rbindlist(lapply(resultados_mc, `[[`, "fila"))
lst_pred_promedio <- lapply(resultados_mc, `[[`, "pred_promedio")
names(lst_pred_promedio) <- paste0("rank", tb_estabilidad$config_rank)

tb_estabilidad

In [ ]:
fwrite(tb_estabilidad, file = "tb_estabilidad.txt", sep = "\t")

cat("\n=== Resumen Monte Carlo (11 configs) ===\n")
print(tb_estabilidad[, .(
  config_rank, AUC, cor_min, overlap_min, overlap_corte_min,
  pasa_cor, pasa_overlap, pasa_corte, estable
)])

cat("\n=== Promedio de metricas sobre las 11 configs ===\n")
print(tb_estabilidad[, .(
  cor_min_prom = mean(cor_min),
  overlap_min_prom = mean(overlap_min),
  overlap_corte_min_prom = mean(overlap_corte_min),
  n_estables = sum(estable)
)])

cat("\n=== Umbrales exigidos ===\n")
cat("cor_min >=", PARAM$montecarlo$min_cor_predicciones, "\n")
cat("overlap_min >=", PARAM$montecarlo$min_overlap_topk, "\n")
cat("overlap_corte_min >=", PARAM$montecarlo$min_overlap_topk, "\n")

cat("\n=== Criterio que mas falla ===\n")
cat("falla cor:      ", sum(!tb_estabilidad$pasa_cor), "/11\n")
cat("falla overlap:  ", sum(!tb_estabilidad$pasa_overlap), "/11\n")
cat("falla corte:    ", sum(!tb_estabilidad$pasa_corte), "/11\n")

tb_estables <- tb_estabilidad[estable == TRUE][order(-AUC)]
cat("\nConfigs estables:", nrow(tb_estables), "de", nrow(tb_estabilidad), "\n")

if (nrow(tb_estables) == 0) {
  cat("\nAVISO: ninguna config paso todos los umbrales.\n")
  cat("Se usara la de mayor cor_min como fallback (ver produccion).\n")
  tb_estabilidad[order(-cor_min)][1, .(config_rank, cor_min, overlap_min, overlap_corte_min, estable)]
} else {
  tb_estables[, .(config_rank, AUC, cor_min, overlap_min, overlap_corte_min, estable)]
}

### 6.3.3 Produccion

#### Produccion, Scoring y Kaggle Submit

Solo las configuraciones que pasaron Monte Carlo generan modelos, predicciones y submits.
Cada config estable usa su corte asignado (rank k → cortes[k]).

##### Modelos y predicciones por config estable


In [ ]:
# helper: prediccion promedio guardada por rank
pred_promedio_de <- function(config_rank) {
  lst_pred_promedio[[paste0("rank", config_rank)]]
}

if (nrow(tb_estables) == 0) {
  warning("Ninguna config paso Monte Carlo. Fallback: mayor cor_min.")
  tb_a_producir <- tb_estabilidad[order(-cor_min)][1]
} else {
  tb_a_producir <- tb_estables
}

for (i in 1:nrow(tb_a_producir)) {
  config <- tb_a_producir[i]
  hiper <- as.list(config[, .(num_leaves, min_data_in_leaf, num_iterations)])

  run <- entrenar_y_predecir(
    hiper, PARAM$semilla_primigenia, dfinal_train, dfuture, campos_buenos
  )

  rank <- config$config_rank
  lgb.save(run$modelo, paste0("modelo_rank", rank, ".txt"))

  tb_importancia <- as.data.table(lgb.importance(run$modelo))
  fwrite(tb_importancia,
    file = paste0("impo_rank", rank, ".txt"),
    sep = "\t"
  )

  if (i == 1) {
    lgb.save(run$modelo, "modelo.txt")
    fwrite(tb_importancia, file = "impo.txt", sep = "\t")
  }
}

In [ ]:
for (i in 1:nrow(tb_a_producir)) {
  config <- tb_a_producir[i]
  rank <- config$config_rank

  tb_pred <- dfuture[, list(numero_de_cliente)]
  tb_pred[, prob := pred_promedio_de(rank)]
  setorder(tb_pred, -prob)

  fwrite(tb_pred,
    file = paste0("prediccion_rank", rank, ".txt"),
    sep = "\t"
  )

  if (i == 1) {
    tb_prediccion <- copy(tb_pred)
    fwrite(tb_prediccion, file = "prediccion.txt", sep = "\t")
  }
}

tb_prediccion

In [ ]:
tb_prediccion

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle **solo para configs estables**.

Cada config estable sube un solo archivo con su corte asignado (rank 1 → 800, rank 2 → 850, ...).


In [ ]:
dir.create("kaggle", showWarnings = FALSE)

for (i in 1:nrow(tb_a_producir)) {
  config <- tb_a_producir[i]
  rank <- config$config_rank
  envios <- min(config$corte_asignado, nrow(dfuture))

  tb_submit <- dfuture[, list(numero_de_cliente)]
  tb_submit[, prob := pred_promedio_de(rank)]
  setorder(tb_submit, -prob)

  tb_submit[, Predicted := 0L]
  tb_submit[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0(
    "./kaggle/KA", PARAM$experimento,
    "_rank", rank, "_", envios, ".csv"
  )

  fwrite(tb_submit[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )

  estable_txt <- if (isTRUE(config$estable)) "SI" else "FALLBACK"
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0(
    "-m 'rank=", rank,
    " envios=", envios,
    " estable=", estable_txt,
    " mc=", PARAM$montecarlo$n_semillas,
    " cor_min=", round(config$cor_min, 4),
    " overlap_min=", round(config$overlap_min, 4),
    " overlap_corte_min=", round(config$overlap_corte_min, 4), "'"
  )

  linea <- paste(comando, competencia, arch, mensaje)
  salida <- system(linea, intern = TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
if (!require("yaml")) install.packages("yaml")
require("yaml")

PARAM$out$montecarlo$resultados <- as.data.frame(tb_estabilidad)
PARAM$out$montecarlo$resumen <- as.list(tb_estabilidad[, .(
  cor_min_prom = mean(cor_min),
  overlap_min_prom = mean(overlap_min),
  overlap_corte_min_prom = mean(overlap_corte_min),
  n_estables = sum(estable)
)])
PARAM$out$montecarlo$configs_estables <- nrow(tb_estables)
PARAM$out$montecarlo$submits_realizados <- as.vector(tb_a_producir$config_rank)

write_yaml(PARAM, file = "PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")